# 04 Treinamento e Avaliação dos Modelos

Cada célula de **avaliação** é **independente**: pode ser executada isoladamente após o kernel reiniciar.  
Células de **treino** (Baseline 2 e Proposta 2) precisam rodar ao menos uma vez para salvar o modelo em disco.

| # | Célula | Pré-requisito |
|---|--------|---------------|
| B1 | Avaliação Baseline 1 | `pt_core_news_lg` instalado |
| P1 | Avaliação Proposta 1 | `data/lexicon/lexicon_filtered.json` |
| B2 | **Treino** Baseline 2 | splits DDsmall |
| B2 | Avaliação Baseline 2 | `models/baseline2/` salvo |
| P2 | **Treino** Proposta 2 | splits DDsmall + `data/processed/ddlarge_pseudo.jsonl` |
| P2 | Avaliação Proposta 2 | `models/proposta2/` salvo |

---
## Baseline 1 `pt_core_news_lg`
Modelo pré-treinado em português, sem adaptação ao domínio.
Labels `PER/LOC/ORG` mapeados para `Person/Location/Organization`.

In [8]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
import spacy
from pathlib import Path
from src.data_loading import load_ddsmall
from src.evaluation import evaluate_ner, predict_records, print_results, PRETRAINED_LABEL_MAP

ROOT = Path('..')
test_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/test/dd_corpus_small_test.json')

# Requer: python -m spacy download pt_core_news_lg
nlp = spacy.load('pt_core_news_lg')

preds  = predict_records(nlp, test_recs, label_map=PRETRAINED_LABEL_MAP)
scores = evaluate_ner(test_recs, preds)
print_results(scores, title='Baseline 1: pt_core_news_lg zero-shot')


=== Baseline 1: pt_core_news_lg zero-shot ===
Label               P      R     F1    TP    FP    FN
----------------------------------------------------
Person          0.408  0.494  0.447   313   455   320
Location        0.409  0.347  0.375   635   918  1197
Organization    0.254  0.149  0.187    88   259   504
----------------------------------------------------
micro           0.388  0.339  0.362
macro F1        0.337


---
## Proposta 1 EntityRuler
Léxico extraído do treino aplicado ao teste como sistema de regras puro.

In [9]:
import json, sys; sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
import spacy
from pathlib import Path
from src.data_loading import load_ddsmall
from src.evaluation import evaluate_ner, predict_records, print_results

ROOT = Path('..')
test_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/test/dd_corpus_small_test.json')

with open(ROOT / 'data/lexicon/lexicon_filtered.json', encoding='utf-8') as f:
    lexicon = json.load(f)

nlp = spacy.blank('pt')
ruler = nlp.add_pipe('entity_ruler', config={'overwrite_ents': True, 'phrase_matcher_attr': 'LOWER'})

# Um padrão por variante de grafia (ex: "polícia" e "policia"), não só a primeira ocorrência
patterns = []
for v in lexicon.values():
    for form in v.get('surface_forms', [v['text']]):
        patterns.append({'label': v['label'], 'pattern': form})
patterns.sort(key=lambda p: len(p['pattern']), reverse=True)

ruler.add_patterns(patterns)
print(f'{len(patterns)} patterns carregados')

nlp.to_disk(ROOT / 'models/proposta1')
print(f'Modelo salvo em models/proposta1/')

preds  = predict_records(nlp, test_recs)
scores = evaluate_ner(test_recs, preds)
print_results(scores, title='Proposta 1: EntityRuler')

2201 patterns carregados
Modelo salvo em models/proposta1/

=== Proposta 1: EntityRuler ===
Label               P      R     F1    TP    FP    FN
----------------------------------------------------
Person          0.551  0.406  0.468   257   209   376
Location        0.830  0.591  0.690  1082   222   750
Organization    0.687  0.671  0.679   397   181   195
----------------------------------------------------
micro           0.739  0.568  0.642
macro F1        0.612


---
## Baseline 2 Treino
NER treinado apenas com o DDsmall. Modelo salvo em `models/baseline2/`.

In [10]:
import sys, warnings
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
warnings.filterwarnings('ignore', category=UserWarning, module='torch')  # silencia aviso sm_120

import spacy
spacy.prefer_gpu()
from pathlib import Path
from src.data_loading import load_ddsmall
from src.training import records_to_examples, train_ner

ROOT = Path('..')

train_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/train/dd_corpus_small_train.json')
val_recs   = load_ddsmall(ROOT / 'data/raw/DDsmall/val/dd_corpus_small_val.json')

nlp_tmp   = spacy.blank('pt')
train_ex  = records_to_examples(train_recs, nlp_tmp)
val_ex    = records_to_examples(val_recs,   nlp_tmp)
print(f'train: {len(train_ex)} exemplos | val: {len(val_ex)} exemplos')

train_ner(train_ex, val_ex, output_path=ROOT / 'models/baseline2', seed=42)

train: 4224 exemplos | val: 201 exemplos
  Epoch   1 | loss= 41166.7 | dev F1=0.5094 ✓
  Epoch   2 | loss= 15717.0 | dev F1=0.7038 ✓
  Epoch   3 | loss= 12497.7 | dev F1=0.7292 ✓
  Epoch   4 | loss= 11147.8 | dev F1=0.7525 ✓
  Epoch   5 | loss=  9749.2 | dev F1=0.7770 ✓
  Epoch   6 | loss=  9025.3 | dev F1=0.7768
  Epoch   7 | loss=  8282.8 | dev F1=0.8042 ✓
  Epoch   8 | loss=  7697.3 | dev F1=0.8087 ✓
  Epoch   9 | loss=  7284.0 | dev F1=0.8112 ✓
  Epoch  10 | loss=  6835.4 | dev F1=0.7954
  Epoch  11 | loss=  6406.7 | dev F1=0.8071
  Epoch  12 | loss=  6241.3 | dev F1=0.7985
  Epoch  13 | loss=  6011.2 | dev F1=0.8122 ✓
  Epoch  14 | loss=  5768.8 | dev F1=0.8037
  Epoch  15 | loss=  5602.9 | dev F1=0.8172 ✓
  Epoch  16 | loss=  5414.0 | dev F1=0.8248 ✓
  Epoch  17 | loss=  5194.4 | dev F1=0.8105
  Epoch  18 | loss=  5100.5 | dev F1=0.8066
  Epoch  19 | loss=  4925.7 | dev F1=0.8100
  Epoch  20 | loss=  4795.0 | dev F1=0.7971
  Epoch  21 | loss=  4766.3 | dev F1=0.8150
  Early stopp

0.8247549019607843

## Baseline 2 Avaliação
Carrega o modelo salvo em `models/baseline2/` e avalia no conjunto de teste.

In [11]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
import spacy
from pathlib import Path
from src.data_loading import load_ddsmall
from src.evaluation import evaluate_ner, predict_records, print_results

ROOT = Path('..')
MODEL_PATH = ROOT / 'models/baseline2'

if not MODEL_PATH.exists():
    print('Modelo não encontrado. Execute a célula de treino do Baseline 2 primeiro.')
else:
    test_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/test/dd_corpus_small_test.json')
    nlp    = spacy.load(MODEL_PATH)
    preds  = predict_records(nlp, test_recs)
    scores = evaluate_ner(test_recs, preds)
    print_results(scores, title='Baseline 2: NER treinado no DDsmall')


=== Baseline 2: NER treinado no DDsmall ===
Label               P      R     F1    TP    FP    FN
----------------------------------------------------
Person          0.749  0.722  0.735   457   153   176
Location        0.834  0.828  0.831  1517   303   315
Organization    0.749  0.750  0.749   444   149   148
----------------------------------------------------
micro           0.800  0.791  0.795
macro F1        0.772


---
## Proposta 2 Treino
NER treinado com DDsmall + amostra do DDlarge pseudo-anotado. Ajuste `N_PSEUDO` conforme o tempo disponível.

In [14]:
import json, random, sys, warnings
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
warnings.filterwarnings('ignore', category=UserWarning, module='torch')  # silencia aviso sm_120

import spacy
spacy.prefer_gpu()
from pathlib import Path
from src.data_loading import load_ddsmall
from src.training import records_to_examples, train_ner

ROOT = Path('..')
# None = dataset completo; defina um inteiro para limitar (ex: 20_000)
N_PSEUDO = None
OUTPUT_PATH = ROOT / 'models/proposta2_full' if N_PSEUDO is None else ROOT / f'models/proposta2_{N_PSEUDO}'

train_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/train/dd_corpus_small_train.json')
val_recs   = load_ddsmall(ROOT / 'data/raw/DDsmall/val/dd_corpus_small_val.json')

pseudo_recs = []
with open(ROOT / 'data/processed/ddlarge_pseudo.jsonl', encoding='utf-8') as f:
    for line in f:
        pseudo_recs.append(json.loads(line))
        if N_PSEUDO is not None and len(pseudo_recs) >= N_PSEUDO:
            break
print(f'Pseudo-registros: {len(pseudo_recs)}')

nlp_tmp    = spacy.blank('pt')
train_ex   = records_to_examples(train_recs,  nlp_tmp)
val_ex     = records_to_examples(val_recs,    nlp_tmp)
pseudo_ex  = records_to_examples(pseudo_recs, nlp_tmp)
combined   = train_ex + pseudo_ex
random.seed(42)  # semente antes do shuffle externo ao train_ner
random.shuffle(combined)
print(f'train: {len(train_ex)} + {len(pseudo_ex)} pseudo = {len(combined)} total | val: {len(val_ex)}')
print(f'Salvando em: {OUTPUT_PATH}')

train_ner(combined, val_ex, output_path=OUTPUT_PATH, seed=42)

Pseudo-registros: 133841
train: 4224 + 133841 pseudo = 138065 total | val: 201
Salvando em: ..\models\proposta2_full
  Epoch   1 | loss=104773.6 | dev F1=0.5588 ✓
  Epoch   2 | loss= 37671.1 | dev F1=0.6023 ✓
  Epoch   3 | loss= 31639.2 | dev F1=0.6431 ✓
  Epoch   4 | loss= 28335.4 | dev F1=0.6296
  Epoch   5 | loss= 26739.2 | dev F1=0.6427
  Epoch   6 | loss= 25245.7 | dev F1=0.6393
  Epoch   7 | loss= 24558.3 | dev F1=0.6431
  Epoch   8 | loss= 23902.6 | dev F1=0.6454 ✓
  Epoch   9 | loss= 23341.3 | dev F1=0.6444
  Epoch  10 | loss= 23253.1 | dev F1=0.6399
  Epoch  11 | loss= 22820.8 | dev F1=0.6431
  Epoch  12 | loss= 22696.5 | dev F1=0.6482 ✓
  Epoch  13 | loss= 22247.3 | dev F1=0.6490 ✓
  Epoch  14 | loss= 22171.0 | dev F1=0.6488


KeyboardInterrupt: 

## Proposta 2 Avaliação
Carrega o modelo salvo em `models/proposta2/` e avalia no conjunto de teste.

In [15]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
import spacy
from pathlib import Path
from src.data_loading import load_ddsmall
from src.evaluation import evaluate_ner, predict_records, print_results

ROOT = Path('..')

# Avalia os dois checkpoints se disponíveis
for label, path in [
    ('Proposta 2 20k pseudo',        ROOT / 'models/proposta2_20000'),
    ('Proposta 2 dataset completo',   ROOT / 'models/proposta2_full'),
]:
    if not path.exists():
        print(f'[{label}] modelo não encontrado em {path}')
        continue
    test_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/test/dd_corpus_small_test.json')
    nlp    = spacy.load(path)
    preds  = predict_records(nlp, test_recs)
    scores = evaluate_ner(test_recs, preds)
    print_results(scores, title=label)


=== Proposta 2 20k pseudo ===
Label               P      R     F1    TP    FP    FN
----------------------------------------------------
Person          0.589  0.441  0.504   279   195   354
Location        0.815  0.658  0.729  1206   273   626
Organization    0.692  0.691  0.692   409   182   183
----------------------------------------------------
micro           0.745  0.620  0.676
macro F1        0.641

=== Proposta 2 dataset completo ===
Label               P      R     F1    TP    FP    FN
----------------------------------------------------
Person          0.535  0.398  0.457   252   219   381
Location        0.828  0.596  0.693  1091   227   741
Organization    0.723  0.640  0.679   379   145   213
----------------------------------------------------
micro           0.745  0.563  0.641
macro F1        0.610


---
## Tabela comparativa
Requer que todos os 4 modelos já tenham sido avaliados nas células acima (variáveis `scores_b1`, etc. em memória), ou lê de `results/metrics.json` se disponível.

In [16]:
import json, sys; sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
import spacy
from pathlib import Path
import pandas as pd
from src.data_loading import load_ddsmall
from src.evaluation import evaluate_ner, predict_records, NER_LABELS, PRETRAINED_LABEL_MAP

ROOT = Path('..')
test_recs = load_ddsmall(ROOT / 'data/raw/DDsmall/test/dd_corpus_small_test.json')

def _load_and_eval(model_path, label_map=None):
    nlp = spacy.load(model_path)
    return evaluate_ner(test_recs, predict_records(nlp, test_recs, label_map))

configs = [
    ('Baseline 1 (zero-shot)',        lambda: _load_and_eval('pt_core_news_lg', PRETRAINED_LABEL_MAP)),
    ('Proposta 1 (EntityRuler)',       lambda: _load_and_eval(ROOT / 'models/proposta1')),
    ('Baseline 2 (DDsmall)',           lambda: _load_and_eval(ROOT / 'models/baseline2')),
    ('Proposta 2 (20k pseudo)',        lambda: _load_and_eval(ROOT / 'models/proposta2_20000')),
    ('Proposta 2 (completo)',          lambda: _load_and_eval(ROOT / 'models/proposta2_full')),
]

rows, all_scores = [], {}
for name, fn in configs:
    path_hint = ROOT / f'models/{name.split("(")[0].strip().lower().replace(" ", "")}'
    try:
        s = fn()
        all_scores[name] = s
        row = {'Modelo': name}
        for lbl in NER_LABELS:
            row[lbl[:3]+'_F1'] = s.get(lbl, {}).get('f1', float('nan'))
        row['micro_F1'] = s['micro']['f1']
        row['macro_F1'] = s['macro']['f1']
        rows.append(row)
    except Exception as e:
        print(f'  {name}: {e}')

df = pd.DataFrame(rows).set_index('Modelo')
print(df.to_string())

with open(ROOT / 'results/metrics.json', 'w', encoding='utf-8') as f:
    json.dump(all_scores, f, ensure_ascii=False, indent=2)
print('\nMétricas salvas em results/metrics.json')

                          Per_F1  Loc_F1  Org_F1  micro_F1  macro_F1
Modelo                                                              
Baseline 1 (zero-shot)    0.4468  0.3752  0.1874    0.3619    0.3365
Proposta 1 (EntityRuler)  0.4677  0.6901  0.6786    0.6424    0.6121
Baseline 2 (DDsmall)      0.7353  0.8308  0.7494    0.7954    0.7718
Proposta 2 (20k pseudo)   0.5041  0.7285  0.6915    0.6763    0.6414
Proposta 2 (completo)     0.4565  0.6927  0.6792    0.6413    0.6095

Métricas salvas em results/metrics.json
